In [1]:
import os
import random

import wandb

api = wandb.Api()

wandb_project = 'learneddb/lcm'


def get_table_df(url: str, label: str):
    print(url)
    artifact = api.artifact(url)
    artifact.download()

    table = artifact.get(label)
    return table.get_dataframe()


if os.path.exists('wandb_df_cache.pkl'):
    import pickle

    with open('wandb_df_cache.pkl', 'rb') as f:
        df_cache = pickle.load(f)
else:
    df_cache = dict()


def get_dataframes_from_run(run: str):
    if run in df_cache:
        print(f'Found run {run} in cache')
        return df_cache[run]

    if wandb.run is None:
        wandb.init()

    df_dict = {}

    instance = api.run(f'{wandb_project}/{run}')

    for key in instance.summary.keys():
        if key.endswith('pred_table'):
            df_dict[key] = get_table_df(f'{wandb_project}/run-{run}-{key}:latest', key)

    df_cache[run] = df_dict

    # save cache to file
    import pickle
    with open('wandb_df_cache.pkl', 'wb') as f:
        pickle.dump(df_cache, f)

    return df_dict

In [5]:
runs = [
    # 'xalwmh52', # job zeroshot
    'iexrv9zv',  # job zeroshot - whintidx
]
run_dict = dict()
for run in runs:
    run_dict[run] = get_dataframes_from_run(run)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: jwehrstein (learneddb). Use `wandb login --relogin` to force relogin


learneddb/lcm/run-iexrv9zv-join_order_job_eval_pred_table:latest


wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  


learneddb/lcm/run-iexrv9zv-join_order_job_light_eval_pred_table:latest


wandb:   1 of 1 files downloaded.  
wandb:   1 of 1 files downloaded.  


In [6]:
run = runs[0]

df = run_dict[run]['join_order_job_eval_pred_table']
grouped_df = df.groupby('sql')

opt_runtimes = []
picked_runtimes = []
opt_plan_instances = []
picked_plan_instances = []

for sql, group in grouped_df:
    min_runtime = group['label'].min()

    # get the picked plan runtime (the one with the lowest prediction)
    picked_plan = group.loc[group['prediction'].idxmin()]
    picked_plan_runtime = picked_plan['label']

    opt_runtimes.append(min_runtime)
    opt_plan_instances.append(group.loc[group['label'].idxmin()])
    picked_runtimes.append(picked_plan_runtime)
    picked_plan_instances.append(picked_plan)

print(f'Optimal runtime: {sum(opt_runtimes)}')
print(f'Picked runtime: {sum(picked_runtimes)}')

Optimal runtime: 156.78919743653387
Picked runtime: 373.2453915817896


In [7]:
from evaluation.analysis_helper import search_plan, create_explain_dalibo_entry
from tabulate import tabulate

# find idx with highest absolute error
absolute_slowdown = [picked - opt for picked, opt in zip(picked_runtimes, opt_runtimes)]
max_idx = absolute_slowdown.index(max(absolute_slowdown))

print(f'Max absolute slowdown: {absolute_slowdown[max_idx]:.2f}s ({max_idx})')

opt_p = opt_plan_instances[max_idx]
picked_p = picked_plan_instances[max_idx]

# find matching plan
opt_plans = search_plan(opt_p['sql'], opt_p['hint'])
picked_plans = search_plan(picked_p['sql'], picked_p['hint'])

assert len(opt_plans) == 1
assert len(picked_plans) == 1

# create dalibo entries
random_id = random.randint(0, 1000)
opt_url = create_explain_dalibo_entry(opt_p['sql'], opt_plans[0]['analyze_plans'][0], f'{random_id}_Optimal')
picked_url = create_explain_dalibo_entry(picked_p['sql'], picked_plans[0]['analyze_plans'][0], f'{random_id}_Picked')

# print stuff
print(opt_p['sql'])
header = ['Val', 'Optimal', 'Picked']
tabulate_rows = [
    ['Runtime', opt_p['label'], picked_p['label']],
    ['Prediction', opt_p['prediction'], picked_p['prediction']],
    ['Absolute Slowdown', '-', absolute_slowdown[max_idx]],
    ['Relative Slowdown', '-', picked_p['label'] / opt_p['label']],
    ['Hint', opt_p['hint'], picked_p['hint']],
    ['Dalibo', opt_url, picked_url]
]
print(tabulate(tabulate_rows, headers=header))

Max absolute slowdown: 27.91s (8)
SELECT MIN(an.name) AS cool_actor_pseudonym, MIN(t.title) AS series_named_after_char FROM aka_name AS an, cast_info AS ci, company_name AS cn, keyword AS k, movie_companies AS mc, movie_keyword AS mk, name AS n, title AS t WHERE cn.country_code ='[us]' AND k.keyword ='character-name-in-title' AND t.episode_nr >= 5 AND t.episode_nr < 100 AND an.person_id = n.id AND n.id = ci.person_id AND ci.movie_id = t.id AND t.id = mk.movie_id AND mk.keyword_id = k.id AND t.id = mc.movie_id AND mc.company_id = cn.id AND an.person_id = ci.person_id AND ci.movie_id = mc.movie_id AND ci.movie_id = mk.movie_id AND mc.movie_id = mk.movie_id;
Val                Optimal                                                                                                                                                                                                                                                                                                                      